<a href="https://colab.research.google.com/github/tripathiosho/ImportantNotebooks/blob/main/Pytorch_Day_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import torch
impor
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from copy import deepcopy

torch.manual_seed(42)

In [ ]:
!wget "https://drive.google.com/uc?export=download&id=1GVhrh2rH6hUunV4Tf7lQoSFsqov9aPjU" -O healthyfime.csv


df = pd.read_csv("healthyfime.csv")

df.replace({"M": 0, "F": 1}, inplace=True)
classes = list(df['class'].unique())
mapping_dict = {ch: i for i, ch in enumerate(sorted(classes, reverse=True))}
df['class'].replace(mapping_dict, inplace=True)

--2026-10-01 02:22:53--  https://drive.google.com/uc?export=download&id=1GVhrh2rH6hUunV4Tf7lQoSFsqov9aPjU
Resolving drive.google.com (drive.google.com)... 74.125.69.138, 74.125.69.101, 74.125.69.113, ...
Connecting to drive.google.com (drive.google.com)|74.125.69.138|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1GVhrh2rH6hUunV4Tf7lQoSFsqov9aPjU&export=download [following]
--2026-10-01 02:22:53--  https://drive.usercontent.google.com/download?id=1GVhrh2rH6hUunV4Tf7lQoSFsqov9aPjU&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.250.152.132, 2607:f8b0:4001:c56::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.250.152.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 761835 (744K) [application/octet-stream]
Saving to: ‘healthyfime.csv’

healthyfime.csv     100%[===================>] 743.98K  --.-KB/s  

/tmp/ipykernel_2083/1294506473.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace({"M": 0, "F": 1}, inplace=True)
/tmp/ipykernel_2083/1294506473.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['class'].replace(mapping_dict, inplace=True)
/tmp/ipykernel_2083/1294506473.py:9: Future

In [ ]:
X, y = df.iloc[:, :-1], df.iloc[:, -1]
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.1, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.1, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.long)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val.values, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.long)

train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

In [ ]:
baseline_model = nn.Sequential(
    nn.Linear(11, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 4)
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    baseline_model.parameters(),
    lr=0.001
)

In [ ]:
n_epochs = 100
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

In [ ]:
total_loss = 0.0
total_examples = 0


for epoch in range(n_epochs):
    baseline_model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad() # 1. Clear gradients left from the previous batch.
        loss = criterion(baseline_model(xb), yb) #Measure the prediction error.
        loss.backward() # Calculate gradients for the model's parameters.
        optimizer.step() #Update those parameters.

    baseline_model.eval()
    with torch.no_grad():
          tr_preds, va_preds = baseline_model(X_train_t), baseline_model(X_val_t)
          history["train_loss"].append(criterion(tr_preds, y_train_t).item())
          history["val_loss"].append(criterion(va_preds, y_val_t).item())
          history["train_acc"].append((tr_preds.argmax(1) == y_train_t).float().mean().item())
          history["val_acc"].append((va_preds.argmax(1) == y_val_t).float().mean().item())

print("Baseline trained. Final val accuracy:", round(history["val_acc"][-1], 4))

Baseline trained. Final val accuracy: 0.7562


In [ ]:
class MultiBranchModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Branch 1: e.g. body-metrics features (11-d)
        self.branch1_fc1 = nn.Linear(11, 16)
        self.branch1_fc2 = nn.Linear(16, 4)
        # Branch 2: e.g. performance-metrics features (5-d)
        self.branch2_fc1 = nn.Linear(5, 4)

        self.out1 = nn.Linear(4, 4)   # e.g. classification head (4 classes)
        self.out2 = nn.Linear(4, 1)   # e.g. regression head (a continuous score)

    def forward(self, x1, x2):
        h1 = torch.relu(self.branch1_fc1(x1))
        h1 = torch.relu(self.branch1_fc2(h1))
        h2 = torch.tanh(self.branch2_fc1(x2))

        out1 = self.out1(h1)   # raw logits for classification
        out2 = self.out2(h2)   # raw value for regression
        return out1, out2


In [ ]:
multi_model = MultiBranchModel()
body_input = torch.randn(8, 11)
performance_input = torch.randn(8, 5)
logits, score = multi_model(body_input, performance_input)
print(logits.shape, score.shape)

torch.Size([8, 4]) torch.Size([8, 1])


In [ ]:
class_targets=torch.zeros(8, dtype=torch.long)

In [ ]:
class_targets

tensor([0, 0, 0, 0, 0, 0, 0, 0])

In [ ]:
score_targets= torch.zeros(8,1, dtype=torch.float32)

In [ ]:
score_targets

tensor([[0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.]])

In [ ]:
classification_loss = nn.CrossEntropyLoss()(logits, class_targets)
regression_loss = nn.MSELoss()(score, score_targets)
combined_loss = classification_loss + 0.1 * regression_loss

multi_model.zero_grad()
combined_loss.backward()

In [ ]:
combined_loss

tensor(1.0727, grad_fn=<AddBackward0>)

In [ ]:
def evaluate_model(model, features, labels, criterion):
    # Select evaluation behavior.
    model.eval()

    # Calculate predictions without recording a gradient graph.
    with torch.no_grad():
        class_scores = model(features)

        loss = criterion(class_scores, labels)

        # Choose the highest-scoring class in each row.
        predicted_classes = class_scores.argmax(dim=1)

        # Compare predictions with the correct labels.
        correct_predictions = predicted_classes == labels

        # True/False → 1/0 → fraction correct.
        accuracy = correct_predictions.float().mean()

    # Convert scalar tensors into ordinary Python numbers.
    return loss.item(), accuracy.item()


for name, Xs, ys in [("Train", X_train_t, y_train_t), ("Val", X_val_t, y_val_t), ("Test", X_test_t, y_test_t)]:
    loss, acc = evaluate_model(baseline_model, Xs, ys, criterion)
    print(f"{name:5s} — loss: {loss:.4f}  accuracy: {acc:.4f}")

Train — loss: 0.5640  accuracy: 0.7688
Val   — loss: 0.5937  accuracy: 0.7562
Test  — loss: 0.5953  accuracy: 0.7649
